## Step 1: Install Required Libraries

Install the packages needed for model loading, quantization, embeddings, and LangChain integration.


In [1]:
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers langchain-core langchain-huggingface langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

## Step 2: Check Runtime and GPU Availability

Inspect Python, PyTorch, CUDA availability, and GPU memory before loading the models.


In [1]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    free_memory, total_memory = torch.cuda.mem_get_info()

    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {total_memory / 1024**3:.2f} GiB")
    print(f"Free VRAM: {free_memory / 1024**3:.2f} GiB")

Python: 3.14.7
PyTorch: 2.14.1+cu130
PyTorch CUDA: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
Total VRAM: 7.96 GiB
Free VRAM: 6.90 GiB


## Step 3: Set the Hugging Face Token

Enter the Hugging Face token and set it for the current session.


In [2]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Hugging Face token: ").strip()

print("Token set for this session.")

Token set for this session.


## Step 4: Load Qwen with 4-Bit Quantization

Load Qwen/Qwen3-4B-Instruct-2507 using NF4 quantization and automatic device placement.


In [2]:
import torch
from transformers import BitsAndBytesConfig, pipeline

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before loading the model.")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

qwen_pipeline = pipeline(
    task="text-generation",
    model="Qwen/Qwen3-4B-Instruct-2507",
    device_map="auto",
    dtype=torch.bfloat16,
    model_kwargs={
        "quantization_config": quantization_config
    }
)

print("Qwen/Qwen3-4B-Instruct-2507 loaded successfully.")

C:\Users\misha\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1013 04:46:47.648000 7336 torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 398/398 [00:09<00:00, 42.39it/s]


Qwen/Qwen3-4B-Instruct-2507 loaded successfully.


## Step 5: Load MedGemma with 4-Bit Quantization

Load google/medgemma-1.5-4b-it using NF4 quantization and automatic device placement.


In [4]:
import torch
from transformers import BitsAndBytesConfig, pipeline

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before loading the model.")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

medgemma_pipe = pipeline(
    task="image-text-to-text",
    model="google/medgemma-1.5-4b-it",
    device_map="auto",
    dtype=torch.bfloat16,
    model_kwargs={
        "quantization_config": quantization_config
    }
)

print("MedGemma-1.5-4b-it loaded successfully.")

Loading weights: 100%|██████████| 883/883 [00:19<00:00, 46.05it/s] 


MedGemma-1.5-4b-it loaded successfully.


## Step 6: Check GPU Memory Usage

Inspect available and used GPU memory after loading the models.


In [5]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    free_memory, total_memory = torch.cuda.mem_get_info()

    print("GPU:", torch.cuda.get_device_name(0))
    print(f"Total VRAM: {total_memory / 1024**3:.2f} GiB")
    print(f"Free VRAM: {free_memory / 1024**3:.2f} GiB")
    print(f"used VRAM: {(total_memory - free_memory) / 1024**3:.2f} GiB")

Python: 3.14.7
PyTorch: 2.14.1+cu130
PyTorch CUDA: 13.0
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
Total VRAM: 7.96 GiB
Free VRAM: 1.02 GiB
used VRAM: 6.94 GiB


## Step 7: Define Model Generation Functions

Define the Qwen and MedGemma response functions and adapt LangChain messages for each model.


In [6]:
import torch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from transformers.utils import logging
from langchain_core.output_parsers import JsonOutputParser


logging.set_verbosity_error()

ROLE_MAP = {
    "system": "system",
    "human": "user",
    "ai": "assistant"
}

medgemma_processor = medgemma_pipe.processor
medgemma_model = medgemma_pipe.model
qwen_tokenizer = qwen_pipeline.tokenizer


# MedGemma
def generate_medgemma_response(prompt_value, max_new_tokens=2000, do_sample=False, temperature=0.9, top_p=0.7, show_thinking=False):
    messages = [
        {
            "role": ROLE_MAP[message.type],
            "content": [{"type": "text", "text": message.content}]
        }
        for message in prompt_value.to_messages()
    ]

    inputs = medgemma_processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(medgemma_model.device)

    input_length = inputs["input_ids"].shape[-1]

    settings = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "temperature": temperature if do_sample else None,
        "top_p": top_p if do_sample else None
    }

    if not do_sample:
        settings["top_k"] = None

    with torch.inference_mode():
        output = medgemma_model.generate(**inputs, **settings)

    response = medgemma_processor.decode(
        output[0][input_length:],
        skip_special_tokens=True
    ).strip()

    if not response:
        raise RuntimeError("The model returned an empty answer.")

    if show_thinking:
        return response

    # Extract the final answer when thinking is hidden.
    if "<unused95>" in response:
        response = response.split("<unused95>", 1)[1].strip()
    elif "<unused94>thought" in response:
        raise RuntimeError(
            "Generation ended before a final answer was produced."
        )

    if not response:
        raise RuntimeError("No final answer was produced.")

    return response


# Qwen
def generate_qwen_response(prompt_value, max_new_tokens=256, do_sample=False, temperature=0.7, top_p=0.8):
    messages = [
        {
            "role": ROLE_MAP[message.type],
            "content": message.content
        }
        for message in prompt_value.to_messages()
    ]

    prompt_text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    settings = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "temperature": temperature if do_sample else None,
        "top_p": top_p if do_sample else None
    }

    if not do_sample:
        settings["top_k"] = None

    output = qwen_pipeline(
        prompt_text,
        return_full_text=False,
        **settings
    )

    response = output[0]["generated_text"].strip()

    if not response:
        raise RuntimeError("Qwen returned an empty answer.")

    return response


print("MedGemma and Qwen connected to LangChain.")

MedGemma and Qwen connected to LangChain.


## Step 8: Test Qwen and MedGemma Responses

Ask both models a simple question to verify that response generation works.


In [7]:
test_prompt = ChatPromptTemplate.from_messages([
    ("system", ""),
    ("human", "{question}")
])

qwen_test_chain = (
    test_prompt
    | RunnableLambda(generate_qwen_response).bind(max_new_tokens=512, do_sample=False)
    | StrOutputParser()
)

medgemma_test_chain = (
    test_prompt
    | RunnableLambda(generate_medgemma_response).bind(max_new_tokens=512, do_sample=False, show_thinking=True)
    | StrOutputParser()
)

test_input = {"question": "What is the capital of France?"}

print("Qwen:", qwen_test_chain.invoke(test_input))
print("MedGemma:", medgemma_test_chain.invoke(test_input))

Qwen: The capital of France is Paris.
MedGemma: <unused94>thought
1.  **Identify the core question:** The user wants to know the capital of France.
2.  **Access knowledge about France:** Recall or look up information about France.
3.  **Identify the capital city:** The capital of France is Paris.
4.  **Formulate the answer:** State the capital clearly and concisely.
5.  **Add context (optional but helpful):** Mention that Paris is also the largest city in France and a major global center for culture, fashion, art, gastronomy, etc. This adds value and provides more information.
6.  **Final Answer:** The capital of France is Paris. Paris is also the largest city in France and a major global center for culture, fashion, art, gastronomy, and tourism.<unused95>The capital of France is **Paris**.


## Step 9: Load the Cleaned Interaction Dataset

Read ddi_cleaned.csv, inspect its shape, and display sample records.


In [8]:
import pandas as pd

df = pd.read_csv("ddi_cleaned.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (191541, 3)


,Drug1_Name,Drug2_Name,Interaction
0,Trioxsalen,Verteporfin,Trioxsalen may increase the photosensitizing a...
1,Aminolevulinic acid,Verteporfin,Aminolevulinic acid may increase the photosens...
2,Titanium dioxide,Verteporfin,Titanium dioxide may increase the photosensiti...
3,Tiaprofenic acid,Verteporfin,Tiaprofenic acid may increase the photosensiti...
4,Cyamemazine,Verteporfin,Cyamemazine may increase the photosensitizing ...


## Step 10: Load the Embedding Model and Chroma Database

Initialize BAAI/bge-small-en-v1.5 and open the persisted ddi_interactions collection.


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from pathlib import Path


DB_PATH = Path(r"ddi_chroma_db_backup\ddi_chroma_db")

if not (DB_PATH / "chroma.sqlite3").is_file():
    raise FileNotFoundError(f"Check database path: {DB_PATH.resolve()}")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 128
    },
    show_progress=False
)

vector_store = Chroma(
    collection_name="ddi_interactions",
    persist_directory=str(DB_PATH),
    embedding_function=embeddings,
    create_collection_if_not_exists=False
)

print("Stored documents:", vector_store._collection.count())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3300.96it/s]


Stored documents: 191541


## Step 11: Configure and Test Retrieval

Retrieve the top record with k=1, format its metadata as JSON, and compare a sample result with the dataset.


In [10]:
import json

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1}
)

def format_context(documents):
    records = [
        {
            "record_id": doc.metadata["record_id"],
            "drug1_name": doc.metadata["drug1_name"],
            "drug2_name": doc.metadata["drug2_name"],
            "interaction": doc.metadata["interaction"]
        }
        for doc in documents
    ]

    return json.dumps(records, ensure_ascii=False, indent=2)

row = df.iloc[0]

q = (
    f"What is the interaction between "
    f"{row['Drug1_Name']} and {row['Drug2_Name']}?"
)

retrieved_documents = retriever.invoke(q)

print("Question:", q)
print("\nReference from df:", row["Interaction"])
print("\nRetrieved context:")
print(format_context(retrieved_documents))

Question: What is the interaction between Trioxsalen and Verteporfin?

Reference from df: Trioxsalen may increase the photosensitizing activities of Verteporfin.

Retrieved context:
[
  {
    "record_id": 1,
    "drug1_name": "Trioxsalen",
    "drug2_name": "Verteporfin",
    "interaction": "Trioxsalen may increase the photosensitizing activities of Verteporfin."
  }
]


## Step 12: Build and Test the Qwen Classification and Extraction Chain

Define the prompt and JSON validation, then classify a question and extract a matching interaction in one Qwen call.


In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser


VERIFY_SYSTEM_PROMPT = """
You are a question reviewer and record extractor for a
drug–drug interaction assistant.

You receive QUESTION and CONTEXT.

1. CHECK QUESTION

Use QUESTION only. Do not obtain medication names from CONTEXT.
Check these conditions in order and stop at the first matching condition:

- If the question is outside drug–drug interactions or taking
  medications together, return "out_of_scope".
  Write one short message in the same language as QUESTION explaining the assistant's scope.

- If the question concerns drug interactions but names only one
  medication, return "out_of_scope".
  Write one short message in the same language as QUESTION asking for the second medication.

- If the question names more than two medications, return "out_of_scope".
  Write one short message in the same language as QUESTION asking the user to select two.

Proceed only if the question asks about an interaction between
exactly two explicitly named medications.
If two names are not provided, return "out_of_scope" and ask for them.

Write explanatory messages in your own words.
Do not combine messages from different conditions.

2. COMPARE WITH CONTEXT

For a valid question, compare both medication names in QUESTION
with drug1_name and drug2_name in CONTEXT.

Accept either order. Ignore letter case and surrounding whitespace.
A retrieved record is not automatically a match.
Do not substitute different medication names.

If both names match, set status to "valid" and copy the matching
interaction field exactly into response.

If they do not match, set status to "valid" and write a short
message in the same language as QUESTION stating that no interaction record for these
two medications was found in the provided context.

OUTPUT

Write all explanatory messages in the same language as QUESTION.
Copy matched interaction text exactly from CONTEXT without translating it.
Return JSON containing only "status" and "response".
status must be "out_of_scope" or "valid".
response must be a string, not a list.
Return JSON only, without Markdown or additional text.

json schema:
{
  "status": "out_of_scope" | "valid",
  "response": "string"
}
""".strip()


verify_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=VERIFY_SYSTEM_PROMPT),
    ("human", "QUESTION:\n{question}\n\nCONTEXT:\n{context}")
])


def validate_output(output):
    if (
        not isinstance(output, dict)
        or set(output) != {"status", "response"}
        or output["status"] not in {"valid", "out_of_scope"}
        or not isinstance(output["response"], str)
    ):
        raise ValueError(f"Invalid model output schema: {output!r}")

    return output


verify_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever | RunnableLambda(format_context)
    }
    | verify_prompt
    | RunnableLambda(generate_qwen_response).bind(
        max_new_tokens=256,
        do_sample=False
    )
    | JsonOutputParser()
    | RunnableLambda(validate_output)
)



q_test = f"ما هو التفاعل الدوائي بين {df.iloc[10000]['Drug1_Name']} و ؟"

response = verify_chain.invoke(q_test)

print("Question:", q_test)
print("Qwen response:", response)

Question: ما هو التفاعل الدوائي بين Canagliflozin و ؟
Qwen response: {'status': 'out_of_scope', 'response': 'السؤال يحتوي على اسم دواء واحد فقط (Canagliflozin) دون تسمية دواء ثانٍ. يُرجى تزويدي بدواء ثانٍ لتحديد التفاعل الدوائي بينه وبين Canagliflozin.'}


## Step 13: Prepare Classification Test Questions

Define 20 labeled questions covering valid interaction requests and out-of-scope cases.


In [16]:
test_data = [
    {"question": "What is the interaction between Trioxsalen and Verteporfin?", "label": "valid"},
    {"question": "Does Verteporfin interact with Trioxsalen?", "label": "valid"},
    {"question": "Can I take Carbamazepine with Doxorubicin?", "label": "valid"},
    {"question": "What happens if I combine Amodiaquine and Celecoxib?", "label": "valid"},
    {"question": "does warfarin interact with ibuprofen", "label": "valid"},
    {"question": "Can I take Metformin with   Cimetidine?", "label": "valid"},
    {"question": "Do Simvastatin and Clarithromycin interact?", "label": "valid"},
    {"question": "I'm taking Digoxin. Can I also take Amiodarone?", "label": "valid"},
    {"question": "What is the interaction between Tirzepatide and Lecanemab?", "label": "valid"},
    {"question": "Can I take Metformin with Donanemab?", "label": "valid"},

    {"question": "I love you.", "label": "out_of_scope"},
    {"question": "What are the symptoms of diabetes?", "label": "out_of_scope"},
    {"question": "What dose of Metformin should I take?", "label": "out_of_scope"},
    {"question": "What are the side effects of Ibuprofen?", "label": "out_of_scope"},
    {"question": "Does Warfarin interact with other medications?", "label": "out_of_scope"},
    {"question": "Can I take Metformin with the other medication?", "label": "out_of_scope"},
    {"question": "Can I take Warfarin, Ibuprofen, and Aspirin together?", "label": "out_of_scope"},
    {"question": "Does Metformin interact with METFORMIN?", "label": "out_of_scope"},
    {"question": "What is the interaction between", "label": "out_of_scope"},
    {"question": "Can I take these two medications together?", "label": "out_of_scope"},
]

## Step 14: Evaluate Classification Accuracy

Calculate overall and per-status accuracy, display responses, and inspect failed cases.


In [17]:
import pandas as pd
from IPython.display import display

results = []

for case in test_data:
    try:
        output = verify_chain.invoke(case["question"])

        results.append({
            "question": case["question"],
            "expected": case["label"],
            "predicted": output["status"],
            "correct": output["status"] == case["label"],
            "response": output["response"],
            "error": None
        })

    except Exception as error:
        results.append({
            "question": case["question"],
            "expected": case["label"],
            "predicted": "error",
            "correct": False,
            "response": None,
            "error": str(error)
        })

results_df = pd.DataFrame(results)

print(f"Overall accuracy: {results_df['correct'].mean():.1%}")
print(f"Correct: {results_df['correct'].sum()} / {len(results_df)}")

accuracy_by_status = (
    results_df.groupby("expected")["correct"]
    .agg(total="count", correct="sum", accuracy="mean")
)
accuracy_by_status["accuracy"] = accuracy_by_status["accuracy"].map(
    lambda value: f"{value:.1%}"
)

print("\nAccuracy by status:")
display(accuracy_by_status)

print("\nAll results:")
display(results_df)

print("\nFailed cases:")
display(results_df[~results_df["correct"]])

Overall accuracy: 95.0%
Correct: 19 / 20

Accuracy by status:


,total,correct,accuracy
expected,,,
out_of_scope,10,9,90.0%
valid,10,10,100.0%



All results:


,question,expected,predicted,correct,response,error
0,What is the interaction between Trioxsalen and...,valid,valid,True,Trioxsalen may increase the photosensitizing a...,None
1,Does Verteporfin interact with Trioxsalen?,valid,valid,True,Trioxsalen may increase the photosensitizing a...,None
2,Can I take Carbamazepine with Doxorubicin?,valid,valid,True,The serum concentration of Carbamazepine can b...,None
3,What happens if I combine Amodiaquine and Cele...,valid,valid,True,The metabolism of Celecoxib can be decreased w...,None
4,does warfarin interact with ibuprofen,valid,valid,True,Warfarin may increase the anticoagulant activi...,None
5,Can I take Metformin with Cimetidine?,valid,valid,True,The serum concentration of Cimetidine can be i...,None
6,Do Simvastatin and Clarithromycin interact?,valid,valid,True,The serum concentration of Clarithromycin can ...,None
7,I'm taking Digoxin. Can I also take Amiodarone?,valid,valid,True,The serum concentration of Amiodarone can be i...,None
8,What is the interaction between Tirzepatide an...,valid,valid,True,No interaction record for these two medication...,None
9,Can I take Metformin with Donanemab?,valid,valid,True,The risk or severity of adverse effects can be...,None



Failed cases:


,question,expected,predicted,correct,response,error
18,What is the interaction between,out_of_scope,valid,False,The serum concentration of Probenecid can be i...,None


## Step 15: Evaluate Interaction Extraction Accuracy

Sample 20 dataset rows and compare the returned interaction with the expected text using exact matching.


In [47]:
import pandas as pd
from IPython.display import display


def evaluate_extraction(df, chain, n=20, seed=42):
    sample = df.sample(n=n, random_state=seed)
    results = []

    for index, row in enumerate(sample.itertuples(index=False), start=1):
        question = (
            f"What is the interaction between "
            f"{row.Drug1_Name} and {row.Drug2_Name}?"
        )
        expected = row.Interaction

        try:
            output = chain.invoke(question)
            predicted = output["response"]

            correct = (
                output["status"] == "valid"
                and isinstance(predicted, str)
                and predicted == expected
            )

            results.append({
                "question": question,
                "status": output["status"],
                "expected_interaction": expected,
                "predicted_interaction": predicted,
                "correct": correct,
                "error": None
            })

        except Exception as error:
            results.append({
                "question": question,
                "status": "error",
                "expected_interaction": expected,
                "predicted_interaction": None,
                "correct": False,
                "error": str(error)
            })

        print(
            f"[{index}/{n}] "
            f"{'PASS' if results[-1]['correct'] else 'FAIL'}"
        )

    results_df = pd.DataFrame(results)

    print(f"\nExact-match accuracy: {results_df['correct'].mean():.1%}")
    print(f"Correct: {results_df['correct'].sum()} / {len(results_df)}")

    with pd.option_context("display.max_colwidth", None):
        print("\nAll results:")
        display(results_df)

        print("\nFailed cases:")
        display(results_df[~results_df["correct"]])

    return results_df


extraction_results = evaluate_extraction(df, verify_chain)

[1/20] PASS
[2/20] PASS
[3/20] PASS
[4/20] PASS
[5/20] PASS
[6/20] PASS
[7/20] PASS
[8/20] PASS
[9/20] PASS
[10/20] PASS
[11/20] PASS
[12/20] PASS
[13/20] PASS
[14/20] PASS
[15/20] PASS
[16/20] PASS
[17/20] PASS
[18/20] PASS
[19/20] PASS
[20/20] PASS

Exact-match accuracy: 100.0%
Correct: 20 / 20

All results:


,question,status,expected_interaction,predicted_interaction,correct,error
0,What is the interaction between Cortisone acetate and Minaprine?,valid,The risk or severity of adverse effects can be increased when Cortisone acetate is combined with Minaprine.,The risk or severity of adverse effects can be increased when Cortisone acetate is combined with Minaprine.,True,None
1,What is the interaction between Amisulpride and Melperone?,valid,Amisulpride may increase the antipsychotic activities of Melperone.,Amisulpride may increase the antipsychotic activities of Melperone.,True,None
2,What is the interaction between Cyclobenzaprine and Lopinavir?,valid,The metabolism of Lopinavir can be decreased when combined with Cyclobenzaprine.,The metabolism of Lopinavir can be decreased when combined with Cyclobenzaprine.,True,None
3,What is the interaction between Nabumetone and Ulobetasol?,valid,The risk or severity of adverse effects can be increased when Nabumetone is combined with Ulobetasol.,The risk or severity of adverse effects can be increased when Nabumetone is combined with Ulobetasol.,True,None
4,What is the interaction between Amitriptyline and Tramadol?,valid,Amitriptyline may increase the neuroexcitatory activities of Tramadol.,Amitriptyline may increase the neuroexcitatory activities of Tramadol.,True,None
5,What is the interaction between Thiopental and Doxazosin?,valid,Thiopental may increase the hypotensive activities of Doxazosin.,Thiopental may increase the hypotensive activities of Doxazosin.,True,None
6,What is the interaction between Primaquine and Haloperidol?,valid,Primaquine may increase the QTc-prolonging activities of Haloperidol.,Primaquine may increase the QTc-prolonging activities of Haloperidol.,True,None
7,What is the interaction between Nilotinib and Digoxin?,valid,Nilotinib may decrease the cardiotoxic activities of Digoxin.,Nilotinib may decrease the cardiotoxic activities of Digoxin.,True,None
8,What is the interaction between Trifluridine and Ouabain?,valid,Trifluridine may decrease the cardiotoxic activities of Ouabain.,Trifluridine may decrease the cardiotoxic activities of Ouabain.,True,None
9,What is the interaction between Apremilast and Tranilast?,valid,The risk or severity of adverse effects can be increased when Apremilast is combined with Tranilast.,The risk or severity of adverse effects can be increased when Apremilast is combined with Tranilast.,True,None



Failed cases:


,question,status,expected_interaction,predicted_interaction,correct,error


## Step 16: Build and Test the Qwen–MedGemma Pipeline

Retrieve once, call Qwen, and invoke MedGemma when its extracted text exactly matches the retrieved interaction; explain it in the question’s language.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser


# Qwen: use the existing prompt without retrieving again.
qwen_chain = (
    verify_prompt
    | RunnableLambda(generate_qwen_response).bind(
        max_new_tokens=256,
        do_sample=False
    )
    | JsonOutputParser()
    | RunnableLambda(validate_output)
)


# MedGemma: receives only the extracted interaction.
MEDGEMMA_SYSTEM_PROMPT = """
Explain the supplied drug interaction clearly and naturally,
using the same language as the user's question.

Use the question to determine the response language.
Base your explanation on the supplied interaction and preserve its meaning.
Keep the explanation concise, ideally 50–80 words.
Do not exceed 80 words.
Return only the explanation.
""".strip()


medgemma_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=MEDGEMMA_SYSTEM_PROMPT),
    ("human", "QUESTION:\n{question}\n\nINTERACTION:\n{interaction}")
])

medgemma_chain = (
    medgemma_prompt
    | RunnableLambda(generate_medgemma_response).bind(
        max_new_tokens=2000,
        do_sample=True,
        show_thinking=False
    )
    | StrOutputParser()
)


def run_pipeline(question):
    # Retrieve once and keep the same record for the comparison.
    documents = retriever.invoke(question)

    # One Qwen call: classification and extraction.
    qwen_result = qwen_chain.invoke({
        "question": question,
        "context": format_context(documents)
    })

    should_call_medgemma = (
        qwen_result["status"] == "valid"
        and bool(documents)
        and qwen_result["response"] == documents[0].metadata["interaction"]
    )

    # Display Qwen's message directly.
    if not should_call_medgemma:
        return {
            "status": qwen_result["status"],
            "response": qwen_result["response"],
            "medgemma_used": False
        }

    # Send only the interaction text to MedGemma.
    explanation = medgemma_chain.invoke({
        "question": question,
        "interaction": qwen_result["response"]
    })

    return {
        "status": "valid",
        "response": explanation,
        "medgemma_used": True
    }


q_test = f"ما هو التفاعل الدوائي بين {df.iloc[10000]['Drug1_Name']} و {df.iloc[10000]['Drug2_Name']}؟"
result = run_pipeline(q_test)

print("Question:", q_test)
print("Status:", result["status"])
print("MedGemma used:", result["medgemma_used"])
print("Response:", result["response"])

Question: ما هو التفاعل الدوائي بين Canagliflozin و Tranylcypromine؟
Status: valid
MedGemma used: True
Response: يمكن أن يزيد Canagliflozin من فعالية Tranylcypromine في خفض مستويات السكر في الدم. هذا يعني أنك قد تحتاج إلى مراقبة مستويات السكر لديك بشكل أكثر دقة عند تناول كلا الدواءين معًا. من المهم استشارة طبيبك قبل البدء بمناول كلا الدواءين لضمان استخدامها بأمان.
